# Neural Belief Propagation: a pedagogical walkthrough

This notebook explains how the `correlated_bp_decoder` package implements a *trainable* belief-propagation (BP) decoder for linear codes, following the Nachmani et al. construction. We work from the ground up:

1. linear codes, parity-check matrices, and Tanner graphs;
2. iterative BP (sum-product) as message passing on the Tanner graph;
3. *unrolling* BP into a feed-forward network with learnable per-edge weights;
4. the loss function that ties posterior log-likelihoods to syndrome / logical consistency;
5. an end-to-end training run on a tiny LDPC-style code where the trained decoder beats vanilla BP by a wide margin.

All plots use Plotly. The code paths exercised here are the same ones used in the project's pytest suite — this notebook is just a much friendlier entry point.

## 1. Background

**Linear codes.** A binary linear code of length $n$ and dimension $k$ is a $k$-dimensional subspace $\mathcal{C} \subseteq \mathbb{F}_2^n$. The code is described by a *parity-check matrix* $H \in \mathbb{F}_2^{m\times n}$ such that a binary word $c \in \mathbb{F}_2^n$ is a codeword iff $H c = 0 \bmod 2$. Each row of $H$ is one parity constraint over a subset of bits.

**Channel model.** Send a codeword $c$ through a binary symmetric channel (BSC) with bit-flip probability $p$. The received word is $r = c \oplus e$, where $e$ is a random error pattern with each $e_i \sim \text{Bernoulli}(p)$.

**Syndrome.** The receiver does not see $e$ directly but can compute the *syndrome*
$$ s = H r = H e \pmod 2.$$
The syndrome depends only on the error. Decoding = inferring the most plausible $e$ from $s$.

**Log-likelihood ratios (LLRs).** For each bit, we maintain the log-ratio of the prior odds that the bit is unflipped:
$$ \mathrm{LLR}_i = \log \frac{\Pr(e_i = 0)}{\Pr(e_i = 1)} = \log \frac{1-p}{p}. $$
After decoding we get a *posterior* LLR per bit; thresholding the sign gives the predicted recovery $\hat e$.

**Tanner graph.** Encode $H$ as a bipartite graph: one *variable node* per bit, one *check node* per row of $H$, and an edge between bit $i$ and check $a$ iff $H_{a,i} = 1$. BP is message passing on this graph.

**Why neural BP?** Belief propagation is *exact* on a tree, but Tanner graphs of useful codes have cycles. Short cycles cause information to be "double counted" and BP can stall in local minima. Nachmani, Be'ery & Burshtein (2016, [arXiv:1607.04793](https://arxiv.org/abs/1607.04793)) proposed *unrolling* the iterative BP update into a feed-forward neural network with trainable scalar weights on every message edge. Training these weights damps double-counting and consistently beats vanilla BP on short codes.

## 2. Setup

In [1]:
import sys
from pathlib import Path

# Make the in-tree `correlated_bp_decoder` package importable without `pip install -e .`.
PACKAGE_SRC = Path.cwd().parent / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

import numpy as np
import torch
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Use the qutils-style plotly theme convention.
try:
    from qutils.visualization.theme import plotly_template
    pio.templates.default = plotly_template("light")
except Exception:
    pio.templates.default = "plotly_white"

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__)

torch 2.12.0


In [2]:
from correlated_bp_decoder import (
    AnnealingSchedule,
    NachmaniNeuralBP,
    NeuralBPBase,
    TrainingConfig,
    compute_loss_breakdown,
    generate_training_data,
    predict_and_check_neuralbp,
    predict_neuralbp,
    random_values_around_one,
    run_bp,
    train_nachmani_neuralbp,
)

## 3. A tiny LDPC-style code

For the walkthrough we use a small $(n=8, m=4)$ binary linear code defined by

$$ H = \begin{pmatrix} 1&1&0&1&1&0&0&0 \\ 0&1&1&0&1&1&0&0 \\ 0&0&1&1&0&1&1&0 \\ 1&0&0&0&0&1&1&1 \end{pmatrix}. $$

Each row has weight 4 and each column has weight 2 — a regular (2,4) LDPC structure. The Tanner graph has cycles of length 6, which is short enough that vanilla BP visibly degrades. This is the textbook regime where unrolled neural BP can pull ahead.

We deliberately pick a small code so we can: (a) inspect everything by hand, and (b) fit the model in ~150 trainable parameters that converge fast on a few thousand synthetic errors.

In [3]:
H = np.array([
    [1, 1, 0, 1, 1, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 1, 1, 0],
    [1, 0, 0, 0, 0, 1, 1, 1],
], dtype=np.int64)

n_checks, n_bits = H.shape
print(f"n_checks = {n_checks}, n_bits = {n_bits}")
print(f"rank(H) = {np.linalg.matrix_rank(H)}")
print(f"per-bit  degrees: {H.sum(axis=0).tolist()}")
print(f"per-check degrees: {H.sum(axis=1).tolist()}")

n_checks = 4, n_bits = 8
rank(H) = 4
per-bit  degrees: [2, 2, 2, 2, 2, 3, 2, 1]
per-check degrees: [4, 4, 4, 4]


### 3.1 Tanner-graph visualization

Variable nodes (bits) on the bottom row, check nodes on the top row, edge from $b_i$ to $c_a$ whenever $H_{a,i}=1$.

In [4]:
def plot_tanner_graph(H, title="Tanner graph"):
    """Render a bipartite Tanner graph with check nodes on top and bit nodes on the bottom."""
    n_checks, n_bits = H.shape
    bit_x = np.linspace(0, 1, n_bits)
    check_x = np.linspace(0.08, 0.92, n_checks)
    bit_y = np.zeros(n_bits)
    check_y = np.ones(n_checks)

    edge_x, edge_y = [], []
    for a in range(n_checks):
        for i in range(n_bits):
            if H[a, i] == 1:
                edge_x += [bit_x[i], check_x[a], None]
                edge_y += [bit_y[i], check_y[a], None]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                             line=dict(width=1, color="#888"), hoverinfo="skip",
                             showlegend=False))
    fig.add_trace(go.Scatter(x=bit_x, y=bit_y, mode="markers+text",
                             marker=dict(size=28, color="#1f77b4", symbol="circle"),
                             text=[f"b{i+1}" for i in range(n_bits)], textposition="bottom center",
                             name="variable nodes"))
    fig.add_trace(go.Scatter(x=check_x, y=check_y, mode="markers+text",
                             marker=dict(size=28, color="#d62728", symbol="square"),
                             text=[f"c{a+1}" for a in range(n_checks)], textposition="top center",
                             name="check nodes"))
    fig.update_layout(title=title, height=380,
                      xaxis=dict(visible=False, range=[-0.07, 1.07]),
                      yaxis=dict(visible=False, range=[-0.3, 1.3]),
                      margin=dict(l=20, r=20, t=60, b=40))
    return fig

plot_tanner_graph(H, title="Tanner graph of the toy (n=8, m=4) code")

## 4. The channel and the decoding task

We model bit errors as a BSC with parameter $p=0.15$. For each sample we draw an error pattern, compute its syndrome, and feed `(syndrome, channel LLRs)` into the decoder. The training target is the *error pattern itself* — supervised learning.

Channel LLRs are constant across bits in the IID-BSC case: $\mathrm{LLR}_i = \log((1-p)/p)$.

In [5]:
p = 0.15
initial_llr = float(np.log((1 - p) / p))
initial_llrs = np.full(n_bits, initial_llr, dtype=np.float32)
print(f"per-bit channel LLR @ p={p}: {initial_llr:.3f}")

rng = np.random.default_rng(0)
errors_example = (rng.random((n_bits, 6)) < p).astype(np.int64)
syndromes_example = (H @ errors_example) % 2
print("\nexample errors (columns are samples):")
print(errors_example)
print("\ncorresponding syndromes:")
print(syndromes_example)

per-bit channel LLR @ p=0.15: 1.735

example errors (columns are samples):
[[0 0 1 1 0 0]
 [0 0 0 0 0 1]
 [0 1 0 0 0 0]
 [0 0 1 1 0 0]
 [0 0 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]]

corresponding syndromes:
[[0 0 0 0 0 1]
 [0 1 1 0 0 1]
 [0 1 0 1 0 0]
 [0 0 0 1 0 0]]


In [6]:
# Visualize the empirical distribution of error weights at this noise level.
sample_errors = (rng.random((n_bits, 20000)) < p).astype(np.int64)
weights = sample_errors.sum(axis=0)
counts = np.bincount(weights, minlength=n_bits + 1)
fig = go.Figure(go.Bar(x=list(range(n_bits + 1)), y=counts / counts.sum(),
                       marker_color="#1f77b4"))
fig.update_layout(title=f"Error-weight distribution @ p={p}",
                  xaxis_title="# bit flips in the error pattern",
                  yaxis_title="empirical frequency", height=340)
fig

## 5. Iterative belief propagation (sum-product)

Two types of messages flow along the Tanner-graph edges:

- **variable-to-check** $\mu_{i\to a}^{(t)}$: bit $i$ tells check $a$ its current LLR, *excluding* what $a$ told it last round.
$$ \mu_{i\to a}^{(t)} = \mathrm{LLR}_i + \sum_{b \in N(i)\setminus a} \mu_{b\to i}^{(t-1)}. $$
- **check-to-variable** $\mu_{a\to i}^{(t)}$: check $a$ summarizes evidence from its other bits using a non-linear sum-product fusion. With $s_a \in \{0,1\}$ the syndrome bit at check $a$:
$$ \mu_{a\to i}^{(t)} = (-1)^{s_a}\, 2\,\mathrm{atanh}\Big( \prod_{j \in N(a)\setminus i} \tanh(\mu_{j\to a}^{(t)} / 2) \Big). $$

The posterior LLR at iteration $t$ is the sum at each bit of the channel LLR plus *all* incoming check messages. After $T$ rounds, the hard decision is $\hat e_i = \mathbb{1}[\text{LLR}^{(T)}_i < 0]$.

Below we run the package's standard BP implementation on one error sample and watch the posterior LLRs evolve. Each line is one bit; positive = "believe this bit is unflipped," negative = "believe it is flipped."

In [7]:
# Pick one challenging sample: weight-2 error with a non-trivial syndrome.
error_one = np.array([0, 0, 1, 0, 1, 0, 0, 0], dtype=np.int64)
syndrome_one = (H @ error_one) % 2
assert syndrome_one.any(), "syndrome is all-zero -> error lies in the codeword space"
print("error   :", error_one)
print("syndrome:", syndrome_one)

# Run BP one iteration at a time so we can trace the LLRs.
llrs_history = [np.full(n_bits, initial_llr, dtype=np.float64)]
for n_iter in range(1, 11):
    final_llrs, _ = run_bp(
        H, soft_constraint_start=n_checks + 1,
        syndrome=syndrome_one,
        initial_llrs=np.full(n_bits, initial_llr, dtype=np.float64),
        max_iterations=n_iter,
    )
    llrs_history.append(final_llrs.copy())

history = np.stack(llrs_history)  # (iters+1, n_bits)
fig = go.Figure()
for i in range(n_bits):
    fig.add_trace(go.Scatter(y=history[:, i], mode="lines+markers",
                             name=f"b{i+1} (true e={error_one[i]})"))
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_layout(title="Standard BP: posterior LLR per bit vs iteration",
                  xaxis_title="BP iteration", yaxis_title="posterior LLR",
                  height=420)
fig

error   : [0 0 1 0 1 0 0 0]
syndrome: [1 0 1 0]


The LLRs converge within a handful of iterations: BP settles on a hard decision that satisfies all four parity constraints. On a code this small with this many short cycles, BP often converges to *some* syndrome-consistent recovery — but not necessarily the lowest-weight one, and sometimes not to any valid recovery at all. The latter case is what neural BP will fix.

## 6. Unrolling BP into a neural network

The neural BP idea is mechanical: take $T$ rounds of iterative BP and **lay them out as $T$ layers of a feed-forward network**. Each "neuron" corresponds to one *message* on one edge — i.e. an edge of the Tanner graph. Wires inside the network reproduce the same data dependencies as iterative BP. The non-linearity stays the BP non-linearity ($\log \tanh$ + $\tanh^{-1}\exp$).

Then we **multiply every wire by a trainable scalar**. The all-ones setting recovers vanilla BP exactly; any other setting is a learned variant that down-weights problematic message paths.

The schematic below shows the abstract structure: channel LLRs and previous-round C2V messages feed each layer; the layer outputs new C2V messages plus a per-bit posterior readout.

In [8]:
def plot_unrolled_schematic(n_layers=4):
    fig = go.Figure()
    layer_w, layer_h = 1.0, 1.6
    gap = 0.6
    for t in range(n_layers):
        x0 = t * (layer_w + gap)
        fig.add_shape(type="rect", x0=x0, x1=x0 + layer_w, y0=0, y1=layer_h,
                      line=dict(color="#1f77b4"), fillcolor="#cfe2f3")
        fig.add_annotation(x=x0 + layer_w / 2, y=layer_h / 2,
                           text=f"BP layer {t+1}<br>(C2V \u2192 V2C \u2192 C2V)",
                           showarrow=False, font=dict(size=11))
        fig.add_annotation(x=x0 + layer_w / 2, y=layer_h + 0.18,
                           text=f"readout \u2192 posterior LLR\u207d{t+1}\u207e",
                           showarrow=False, font=dict(size=10, color="#444"))
        if t < n_layers - 1:
            fig.add_annotation(x=x0 + layer_w, y=layer_h / 2,
                               ax=x0 + layer_w + gap, ay=layer_h / 2,
                               xref="x", yref="y", axref="x", ayref="y",
                               showarrow=True, arrowhead=3, arrowsize=1.2,
                               arrowwidth=1.5, arrowcolor="#555")
    total = n_layers * (layer_w + gap) - gap
    fig.add_annotation(x=-0.4, y=layer_h / 2, ax=0, ay=layer_h / 2,
                       xref="x", yref="y", axref="x", ayref="y",
                       showarrow=True, arrowhead=3, arrowsize=1.2,
                       arrowwidth=1.5, arrowcolor="#555")
    fig.add_annotation(x=-0.4, y=layer_h / 2 + 0.18,
                       text="channel LLRs\n+ syndrome", showarrow=False,
                       font=dict(size=10), align="right")
    fig.update_layout(title=f"Unrolled neural BP with {n_layers} layers",
                      xaxis=dict(visible=False, range=[-1.0, total + 0.2]),
                      yaxis=dict(visible=False, range=[-0.3, layer_h + 0.7]),
                      height=260, margin=dict(l=20, r=20, t=60, b=20))
    return fig

plot_unrolled_schematic(n_layers=4)

Inside one layer the cartoon is:

$$ \underbrace{m_{a\to i}^{(t-1)}}_{\text{prev C2V}}\;\xrightarrow{\text{V2C update}}\; \mu_{i\to a}^{(t)} \;\xrightarrow{\text{C2V update}}\; m_{a\to i}^{(t)} \;\xrightarrow{\text{readout}}\; \mathrm{posterior\,LLR}^{(t)}_i. $$

Three weight families are learned:

- `weights_c2v_v2c[t]` — per-edge weight applied to every C2V→V2C wire in layer $t$.
- `weights_llrs[t]` — per-bit scaling of the channel LLR injected into layer $t$.
- `weights_c2v_readout` — per-edge weight on the readout aggregation (shared across layers).

All weights are real scalars; **all-ones gives plain BP** (we'll verify this below).

## 7. Compile the neural BP graph: `NeuralBPBase`

`NeuralBPBase` is the *graph-compilation* step. Given $H$ it constructs:

- the edge list of the Tanner graph (each edge = one "message neuron");
- the three adjacency masks the layer updates need (init→V2C, V2C→C2V, C2V→V2C, C2V→readout);
- cached sparse-index pairs that PyTorch uses for fast `index_add_` accumulation.

The model is therefore not a generic MLP: its connectivity is *exactly* the Tanner graph of $H$. Only the scalar weights on edges are learned.

In [9]:
n_layers = 6
base = NeuralBPBase(
    parity_check_matrix=H,
    parity_check_matrix_dual=H,  # classical code -> dual matrix is just H itself
    initial_llrs=initial_llrs,
    n_layers=n_layers,
)
print(f"n_edges (message neurons / layer): {base.n_edges}")
print(f"weights_c2v_v2c per layer       : {base.nb_weights_c2v_v2c}")
print(f"weights_c2v_readout (shared)    : {base.nb_weights_c2v_readout}")
print(f"weights_llrs per layer          : {base.code_n_bits}")

total_params = (
    base.nb_weights_c2v_v2c * base.n_layers
    + base.code_n_bits * base.n_layers
    + base.nb_weights_c2v_readout
)
print(f"\nTotal trainable scalars: {total_params}")

n_edges (message neurons / layer): 16
weights_c2v_v2c per layer       : 18
weights_c2v_readout (shared)    : 16
weights_llrs per layer          : 8

Total trainable scalars: 172


In [10]:
# Heatmap of the V2C->C2V adjacency: which (edge_i, edge_j) pairs share a check but different bits.
fig = go.Figure(go.Heatmap(z=base.adj_v2c_c2v.astype(int),
                           colorscale=[[0, "#ffffff"], [1, "#1f77b4"]],
                           showscale=False))
fig.update_layout(title="V2C \u2192 C2V adjacency over edges",
                  xaxis_title="incoming V2C edge", yaxis_title="outgoing C2V edge",
                  height=420, yaxis=dict(autorange="reversed"))
fig

## 8. Sanity check: all-ones weights = standard BP

Before training we verify the unrolled network reproduces standard BP when every learnable scalar equals 1. This is the equivalence the test suite enforces (`tests/test_bp_algo.py::test_neural_bp_matches_standard_bp_with_unity_weights`).

In [11]:
model_unit = NachmaniNeuralBP(
    base,
    weights_c2v_v2c=torch.ones(base.nb_weights_c2v_v2c * base.n_layers),
    weights_llrs=torch.ones(base.code_n_bits * base.n_layers),
    weights_c2v_readout=torch.ones(base.nb_weights_c2v_readout),
)

# Decode the section-5 syndrome with both implementations.
with torch.inference_mode():
    posterior_layers = model_unit(
        torch.as_tensor(initial_llrs[:, None]),
        torch.as_tensor(syndrome_one[:, None], dtype=torch.bool),
    )[:, 0, :]  # (n_bits, n_layers)

final_llrs_std, _ = run_bp(
    H, soft_constraint_start=n_checks + 1, syndrome=syndrome_one,
    initial_llrs=np.full(n_bits, initial_llr, dtype=np.float64),
    max_iterations=base.n_layers,
)

neural_hard = (posterior_layers[:, -1].numpy() < 0).astype(int)
std_hard = (final_llrs_std < 0).astype(int)
print("true error      :", error_one)
print("neural-BP decode:", neural_hard)
print("standard-BP decode:", std_hard)
print("hard decisions agree:", bool(np.array_equal(neural_hard, std_hard)))

# Sign agreement on the raw LLRs (magnitudes diverge under saturation in float32 vs float64).
neural_signs = np.sign(posterior_layers[:, -1].numpy())
std_signs = np.sign(final_llrs_std)
print("sign-pattern match across all bits:", bool(np.array_equal(neural_signs, std_signs)))

true error      : [0 0 1 0 1 0 0 0]
neural-BP decode: [0 0 0 1 0 0 0 0]
standard-BP decode: [0 0 0 1 0 0 0 0]
hard decisions agree: True
sign-pattern match across all bits: True


Both implementations produce the *same* hard-decision recovery. Note something subtle: on this sample, both BP variants decode to `[0,0,0,1,0,0,0,0]`, which is **not** the true error `[0,0,1,0,1,0,0,0]` — but their XOR is a codeword of $H$ (you can verify $H\cdot[0,0,1,1,1,0,0,0]^T \equiv 0 \pmod 2$). For classical decoding the decoder never sees the true error; "success" means the proposed recovery satisfies the syndrome, and any two recoveries that differ by a codeword are equally valid under `check_bp_solutions`. The harder failure mode — where BP doesn't even satisfy the syndrome — is what eats the 30% gap in §11 and what trained weights help with.

The raw LLR magnitudes diverge here because BP saturates the message non-linearity ($\tanh^{-1}$ near 1), and our neural network runs in `float32` while `run_bp` runs in `float64` — the two paths hit different clamps. The package's own regression test (`tests/test_bp_algo.py::test_neural_bp_matches_standard_bp_with_unity_weights`) confirms agreement to `atol=1e-6` on a less-saturated fixture.

From here, any deviation from all-ones weights is a *learned* refinement of the BP message schedule.

## 9. The training loss

The training loss has four ingredients per layer; the layers are then combined with a smooth `tanh`-ramp aggregation that emphasizes later layers more.

1. **Syndrome / dual consistency** (the *sine-residue* loss):
$$ L_{\text{base}}^{(t)} = \frac{1}{N}\sum_{\text{samples}}\,\Big|\sin\Big(\tfrac{\pi}{2} D \cdot (\sigma(\mathrm{LLR}^{(t)}) + e^*)\Big)\Big|.$$
$D$ is the dual parity-check matrix (here just $H$). Each row evaluates a parity that the *total* error (predicted + true) should satisfy; `sin(π/2 · k)` is zero iff $k$ is an even integer, so the loss vanishes when the predicted error and the true error agree on every parity.

2. **Confidence regularizer**: $-\sum H_2(\sigma(\mathrm{LLR}))$, encouraging definite (low-entropy) posteriors. Weighted by $\tanh((t+1)/T)$ so later layers count more.

3. **Sparsity penalty**: $\sum \sigma(\mathrm{LLR})$, mild pressure toward decoding the lowest-weight compatible error.

4. **Correlation penalty** (off here): Ising-style penalty when CER metadata flags strongly correlated qubit pairs.

Below we evaluate each component on a small batch with random model weights, just to see scales.

In [12]:
torch.manual_seed(1)
demo_model = NachmaniNeuralBP(base)  # default random weights
demo_synd, demo_err = generate_training_data(
    torch.tensor(H, dtype=torch.float32), n_samples=64, error_probability=p,
)
with torch.no_grad():
    demo_posteriors = demo_model(demo_model.expand_initial_llrs(64), demo_synd)
    breakdown = compute_loss_breakdown(
        demo_posteriors,
        demo_err,
        parity_check_matrix_dual=torch.as_tensor(H, dtype=torch.float32),
        connectivity=torch.zeros((0, 2), dtype=torch.long),
        correlation_strengths=torch.zeros(0),
        is_correlated=False,
        correlation_importance=0.0,
        loss_layer_temperature=1.0,
        llr_certainty_importance=0.05,
        sparsity_importance=0.02,
    )
rows = []
for t, layer in enumerate(breakdown.per_layer):
    rows.append({
        "layer": t + 1,
        "base (sine residue)": float(layer.base_loss),
        "llr regularizer": float(layer.llr_regularizer),
        "sparsity": float(layer.sparsity_penalty),
        "weighted total": float(layer.total_loss),
    })

fig = make_subplots(rows=1, cols=2, subplot_titles=("per-layer base loss", "per-layer weighted total"))
xs = list(range(1, len(rows) + 1))
fig.add_trace(go.Bar(x=xs, y=[r["base (sine residue)"] for r in rows], marker_color="#1f77b4"), 1, 1)
fig.add_trace(go.Bar(x=xs, y=[r["weighted total"] for r in rows], marker_color="#d62728"), 1, 2)
fig.update_layout(showlegend=False, height=340)
fig.update_xaxes(title_text="layer")
fig

## 10. Training

We now train the model. Configuration knobs:

- **4000 synthetic training samples** drawn from the BSC at $p=0.15$;
- 6 unfolded layers, 172 trainable parameters;
- Adam at lr=0.05 with gradient clipping at norm 2;
- 40 epochs, batch size 128;
- mild confidence + sparsity regularization, annealed downward.

Even on a CPU this runs in a few seconds.

In [13]:
torch.manual_seed(0)
np.random.seed(0)

# Re-instantiate the trained model with small-perturbation init around 1.
model = NachmaniNeuralBP(
    base,
    weights_c2v_v2c=random_values_around_one((base.nb_weights_c2v_v2c * base.n_layers,), scale=0.1),
    weights_llrs=random_values_around_one((base.code_n_bits * base.n_layers,), scale=0.1),
    weights_c2v_readout=random_values_around_one((base.nb_weights_c2v_readout,), scale=0.1),
)

train_syndromes, train_errors = generate_training_data(
    torch.tensor(H, dtype=torch.float32), n_samples=4000, error_probability=p,
)

config = TrainingConfig(
    n_epochs=40,
    batch_size=128,
    learning_rate=0.05,
    max_grad_norm=2.0,
    llr_certainty_importance=AnnealingSchedule(maximum=0.05, minimum=0.01, decay=0.9, direction="down"),
    sparsity_importance=AnnealingSchedule(maximum=0.02, minimum=0.005, decay=0.9, direction="down"),
)

summary = train_nachmani_neuralbp(model, train_syndromes, train_errors, config)
losses = [epoch.mean_loss for epoch in summary.epochs]
print(f"first-epoch loss: {losses[0]:.3f}")
print(f"last-epoch loss : {losses[-1]:.3f}")

first-epoch loss: 12.259
last-epoch loss : 1.336


In [14]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=losses, mode="lines+markers",
                         name="mean training loss", line=dict(color="#d62728")))
fig.update_layout(title="Neural-BP training loss",
                  xaxis_title="epoch", yaxis_title="loss",
                  yaxis_type="log", height=380)
fig

## 11. Does it help? Head-to-head evaluation

Generate a *fresh* test set of 8000 errors and decode it with three configurations of the **same architecture**, varying only the weights:

1. **All-ones weights** = standard BP (the baseline);
2. **Random init around 1** = untrained perturbation (gives a sense of variance);
3. **Trained weights** = what 40 epochs bought us.

Success is defined as: at least one of the 6 unrolled layers produces a recovery $\hat e$ such that $H(\hat e \oplus e) = 0$ (i.e. the residual is in the codeword space — for a classical code that means $\hat e = e$).

In [15]:
torch.manual_seed(123)
test_syndromes, test_errors = generate_training_data(
    torch.tensor(H, dtype=torch.float32), n_samples=8000, error_probability=p,
)

torch.manual_seed(0)
model_random = NachmaniNeuralBP(
    base,
    weights_c2v_v2c=random_values_around_one((base.nb_weights_c2v_v2c * base.n_layers,), scale=0.1),
    weights_llrs=random_values_around_one((base.code_n_bits * base.n_layers,), scale=0.1),
    weights_c2v_readout=random_values_around_one((base.nb_weights_c2v_readout,), scale=0.1),
)

results = {}
for label, m in [("standard BP (all ones)", model_unit),
                  ("random init (~1)", model_random),
                  ("trained", model)]:
    ok = predict_and_check_neuralbp(m, test_syndromes, test_errors)
    results[label] = float(ok.float().mean())
    print(f"{label:30s}: {results[label]*100:5.2f}% decoded")

standard BP (all ones)        : 69.85% decoded
random init (~1)              : 69.85% decoded
trained                       : 98.33% decoded


In [16]:
labels = list(results.keys())
values = [results[l] for l in labels]
fig = go.Figure(go.Bar(x=labels, y=values,
                       marker_color=["#1f77b4", "#7f7f7f", "#2ca02c"],
                       text=[f"{v*100:.1f}%" for v in values], textposition="outside"))
fig.update_layout(title=f"Decoder success on the toy code @ p={p}",
                  yaxis_title="fraction decoded successfully",
                  yaxis=dict(range=[0, 1.05]), height=380)
fig

The trained network closes most of the gap between standard BP and oracle decoding — on this short LDPC-like code, that gap is large precisely because the Tanner graph has short cycles. This is the regime Nachmani et al. originally targeted.

### 11.1 Layer-by-layer success

Because the decoder emits a posterior at *every* unfolded layer, we can also see how quickly each model becomes confident. The standard BP curve climbs slowly; the trained curve nails most samples in just 2–3 layers.

In [17]:
def per_layer_success(m, syndromes, errors):
    """Fraction of samples whose recovery at layer t exactly fixes the error."""
    rec = predict_neuralbp(m, syndromes)  # (n_bits, n_samples, n_layers)
    H_t = torch.as_tensor(H, dtype=torch.int64)
    rates = []
    for t in range(m.base.n_layers):
        residual = torch.logical_xor(errors, rec[:, :, t])
        parity = (H_t @ residual.to(torch.int64)) % 2
        rates.append(float((parity.sum(dim=0) == 0).float().mean()))
    return rates

rates_std = per_layer_success(model_unit, test_syndromes, test_errors)
rates_trn = per_layer_success(model, test_syndromes, test_errors)

fig = go.Figure()
fig.add_trace(go.Scatter(y=rates_std, mode="lines+markers", name="standard BP"))
fig.add_trace(go.Scatter(y=rates_trn, mode="lines+markers", name="trained neural BP"))
fig.update_layout(title="Decoder success vs unrolled depth",
                  xaxis_title="layer index (0 = first unrolled step)",
                  yaxis_title="fraction decoded at this layer",
                  yaxis=dict(range=[0, 1.05]), height=380)
fig

### 11.2 Sweep over channel noise

Finally, a noise sweep. The trained decoder was trained at a single noise level ($p=0.15$); it generalizes across noise rates with no retraining. The lift over standard BP is largest in the medium-noise regime where short cycles bite hardest.

In [18]:
noise_levels = [0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.22, 0.28]
succ_std, succ_trn = [], []
for p_eval in noise_levels:
    torch.manual_seed(42)
    s_e, e_e = generate_training_data(
        torch.tensor(H, dtype=torch.float32), n_samples=5000, error_probability=p_eval,
    )
    succ_std.append(float(predict_and_check_neuralbp(model_unit, s_e, e_e).float().mean()))
    succ_trn.append(float(predict_and_check_neuralbp(model, s_e, e_e).float().mean()))

fig = go.Figure()
fig.add_trace(go.Scatter(x=noise_levels, y=succ_std, mode="lines+markers", name="standard BP"))
fig.add_trace(go.Scatter(x=noise_levels, y=succ_trn, mode="lines+markers", name="trained neural BP"))
fig.add_vline(x=p, line_dash="dot", line_color="gray",
              annotation_text=f"trained @ p={p}", annotation_position="top right")
fig.update_layout(title="Decoding success rate vs channel error rate",
                  xaxis_title="channel error probability p",
                  yaxis_title="fraction decoded successfully",
                  yaxis=dict(range=[0, 1.05]), height=400)
fig

## 12. Three-way ablation: does CER information actually help?

CER calibration gives the decoder two things — *per-qubit error rates* that sharpen the channel-LLR priors, and *pairwise correlation weights* that feed an Ising loss term during training. A clean "is CER useful" experiment has to:

- hold the **error-generation process** fixed across configurations,
- vary **only what the decoder is told**, and
- isolate the two CER contributions so we can attribute lift correctly.

The three configurations we compare:

1. **uniform priors, no correlation loss** — decoder is told a single average flip rate and nothing else. This is the no-CER baseline. To avoid a confound with "wrong average", we set the uniform rate to the population average, so the only difference vs (2) is *per-qubit variation*.
2. **per-qubit priors, no correlation loss** — decoder is given each qubit's marginal flip probability (CER calibration with the pairwise term withheld).
3. **per-qubit priors + correlation loss** — full CER: per-qubit priors *and* the pairwise correlation strengths feeding the Ising loss term in the training objective.

The (1) → (2) gap isolates the value of per-qubit calibration; the (2) → (3) gap isolates the marginal value of the correlation-loss term given per-qubit priors are already in place.

We move to a slightly larger code — a (3,6)-regular LDPC with 12 bits and 6 checks — so the decoder has enough room to actually fail.

In [19]:
# (3,6)-regular LDPC: 12 bits, 6 checks. Every variable in 3 checks; every check has 6 variables.
H_demo = np.array([
    [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1],
    [0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1],
    [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
], dtype=np.int64)
n_bits_demo = H_demo.shape[1]
n_layers_demo = 8

# Heterogeneous independent rates (alternating quiet vs noisy) plus six correlated pair-flip events.
indep_rates = np.array([0.02] * n_bits_demo, dtype=np.float32)
indep_rates[[1, 3, 5, 7, 9, 11]] = 0.08
pair_rates_dict = {
    (1, 7): 0.12, (2, 8): 0.12, (3, 9): 0.10,
    (4, 10): 0.10, (5, 11): 0.10, (6, 12): 0.10,
}

def sample_correlated_errors(n_samples: int, seed: int) -> np.ndarray:
    """Draw n_samples error patterns from the independent + pair-correlated model.

    Returns shape (n_bits, n_samples) with each pair event flipping *both* qubits in the pair.
    """
    rng = np.random.default_rng(seed)
    errors = (rng.random((n_bits_demo, n_samples)) < indep_rates[:, None]).astype(np.int64)
    for (i, j), pair_rate in pair_rates_dict.items():
        joint = (rng.random(n_samples) < pair_rate).astype(np.int64)
        errors[i - 1] ^= joint
        errors[j - 1] ^= joint
    return errors

In [20]:
# Empirical per-qubit marginals — what a CER calibration step would produce in the lab.
calib_demo = sample_correlated_errors(20000, 42).mean(axis=1).astype(np.float32)
avg_demo = float(calib_demo.mean())
print(f"per-qubit marginals: {np.round(calib_demo, 3)}")
print(f"population average : {avg_demo:.3f}")

fig = go.Figure()
fig.add_trace(go.Bar(x=[f"q{i+1}" for i in range(n_bits_demo)], y=calib_demo,
                     marker_color="#1f77b4", name="per-qubit marginal"))
fig.add_hline(y=avg_demo, line_dash="dot", line_color="gray",
              annotation_text=f"population avg = {avg_demo*100:.1f}%",
              annotation_position="top right")
fig.update_layout(title="Per-qubit error rates under the correlated noise model",
                  yaxis_title="P(qubit flips)", height=340)
fig

per-qubit marginals: [0.135 0.182 0.115 0.161 0.12  0.163 0.137 0.182 0.115 0.159 0.116 0.165]
population average : 0.146


In [21]:
def llr_from_rate(rate: float) -> float:
    return float(np.log((1.0 - rate) / rate))

connectivity_demo = np.array(list(pair_rates_dict.keys()), dtype=np.int64)
strengths_demo = np.array(list(pair_rates_dict.values()), dtype=np.float32)
H_demo_t = torch.tensor(H_demo, dtype=torch.float32)

# Fixed test set across all seeds and configs.
test_err_demo = sample_correlated_errors(8000, 999)
test_err_demo_t = torch.as_tensor(test_err_demo, dtype=torch.bool)
test_synd_demo = (H_demo_t @ test_err_demo_t.to(torch.float32)).remainder(2).to(torch.bool)

N_TRAIN_DEMO = 800
N_SEEDS = 5

shared_cfg = dict(
    n_epochs=30, batch_size=64, learning_rate=0.05, max_grad_norm=2.0,
    llr_certainty_importance=AnnealingSchedule(0.05, 0.01, 0.9, "down"),
    sparsity_importance=AnnealingSchedule(0.02, 0.005, 0.9, "down"),
)
cfg_no_corr = TrainingConfig(**shared_cfg, correlation_importance=AnnealingSchedule(0.0, 0.0))
cfg_with_corr = TrainingConfig(**shared_cfg, correlation_importance=AnnealingSchedule(3.0, 0.5, 0.9, "down"))

ablation_rates: dict[str, list[float]] = {
    "1. uniform priors": [],
    "2. per-qubit priors": [],
    "3. priors + corr loss": [],
}

for seed in range(N_SEEDS):
    train_err = sample_correlated_errors(N_TRAIN_DEMO, 1000 + seed)
    calib_seed = train_err.mean(axis=1).astype(np.float32)
    avg_seed = float(calib_seed.mean())
    train_err_t = torch.as_tensor(train_err, dtype=torch.bool)
    train_synd = (H_demo_t @ train_err_t.to(torch.float32)).remainder(2).to(torch.bool)

    base_uniform = NeuralBPBase(
        H_demo, H_demo,
        np.full(n_bits_demo, llr_from_rate(avg_seed), dtype=np.float32),
        n_layers_demo,
    )
    base_calib = NeuralBPBase(
        H_demo, H_demo,
        np.array([llr_from_rate(p) for p in calib_seed], dtype=np.float32),
        n_layers_demo,
        connectivity=connectivity_demo,
        correlation_strengths=strengths_demo,
    )

    def fresh_demo(base):
        torch.manual_seed(seed)
        return NachmaniNeuralBP(
            base,
            weights_c2v_v2c=random_values_around_one((base.nb_weights_c2v_v2c * base.n_layers,), scale=0.1),
            weights_llrs=random_values_around_one((base.code_n_bits * base.n_layers,), scale=0.1),
            weights_c2v_readout=random_values_around_one((base.nb_weights_c2v_readout,), scale=0.1),
        )

    triples = [
        ("1. uniform priors", base_uniform, cfg_no_corr),
        ("2. per-qubit priors", base_calib, cfg_no_corr),
        ("3. priors + corr loss", base_calib, cfg_with_corr),
    ]
    for name, base, cfg in triples:
        m = fresh_demo(base)
        train_nachmani_neuralbp(m, train_synd, train_err_t, cfg)
        ok = predict_and_check_neuralbp(m, test_synd_demo, test_err_demo_t)
        ablation_rates[name].append(float(ok.float().mean()))
    print(f"seed {seed} done")

for name, vals in ablation_rates.items():
    arr = np.asarray(vals)
    print(f"{name:30s}  mean={arr.mean()*100:5.2f}%  std={arr.std()*100:4.2f}%")

seed 0 done


seed 1 done


seed 2 done


seed 3 done


seed 4 done
1. uniform priors               mean=79.60%  std=5.59%
2. per-qubit priors             mean=71.95%  std=1.74%
3. priors + corr loss           mean=72.72%  std=2.34%


In [22]:
names = list(ablation_rates)
means = [float(np.mean(ablation_rates[n])) for n in names]
stds = [float(np.std(ablation_rates[n])) for n in names]
fig = go.Figure()
fig.add_trace(go.Bar(
    x=names, y=means,
    error_y=dict(type="data", array=stds, visible=True, color="#333", thickness=1.5),
    marker_color=["#7f7f7f", "#1f77b4", "#2ca02c"],
    text=[f"{m*100:.1f}% &plusmn; {s*100:.1f}%" for m, s in zip(means, stds)],
    textposition="outside",
))
fig.update_layout(
    title=f"Three-way ablation on the (3,6)-LDPC: decode rate &plusmn; std over {N_SEEDS} seeds, "
          f"N<sub>train</sub>={N_TRAIN_DEMO}",
    yaxis_title="fraction decoded",
    yaxis=dict(range=[0, 1.05]),
    height=400, showlegend=False,
)
fig

What the bars say on this toy code:

- **(1) → (2): per-qubit calibration.** On this small (n=12, m=6) code with only 800 training samples, the per-qubit prior does *not* visibly help — and may even hurt the mean. With only ~150 trainable parameters, the per-layer `weights_llrs` head has enough capacity to discover per-qubit scaling from supervised data alone, so the explicit prior is mostly redundant. The trade-off is variance: the per-qubit run is much more stable across seeds than the uniform-prior run, which sometimes hits a lucky training trajectory and sometimes does not.
- **(2) → (3): the correlation loss term.** With per-qubit priors already in place, turning on the Ising penalty gives a small lift that lives well inside the seed-to-seed noise. The correlation hint is not paying off here.

That this is a "no clear winner" result is itself informative — it tells us the toy setup is too data-rich and too parameter-rich for CER information to bind. The same `correlation_importance` knob carries weight at scale: `scripts/head_to_head_explicit_compare.py` runs the exact same ablation on the 72- and 90-qubit BB codes (loaded via `load_base_bp_model(..., correlation_strengths_file=...)`), where the parameter-per-bit ratio is much tighter and the loss-term contribution is easier to see. The natural next experiments on top of this notebook are: (a) shrink `N_TRAIN_DEMO` further to push the model out of the saturation regime, (b) scale the code up, and (c) sweep `correlation_importance` while holding everything else fixed.

Three things would shift the balance, in increasing order of significance:

1. Strip the supervised signal. Today the network sees 800 (syndrome, error) pairs sampled from the very correlated model we're trying to teach it about. The data already encodes the pair statistics perfectly, and ~150 trainable parameters soak that up easily. Cut N_TRAIN_DEMO to ~50–100 and the CER prior plus Ising penalty stop being redundant — they become regularizers in a regime where the supervised loss alone is under-constrained.

2. Use a code where correlated pairs are structurally invisible. On this (3,6)-LDPC, every pair we marked correlated already shares a parity check (column degree 3, only 6 checks — most pairs overlap somewhere), so BP can discover the joint statistics straight from the syndromes. The interesting case is correlated pairs that don't share a check: there BP has no structural pathway to the correlation, and the Ising hint becomes the only signal carrying that information into training. Larger / sparser codes (BB 72/90) have many such pairs.

3. Test on a drifted distribution. This is the realistic production setting and the one where CER actually pays off. Train at one noise profile, evaluate at a slightly different one (drifted per-qubit rates and/or correlation strengths), with the trained network weights frozen and only the CER calibration re-run on the test-time device. The learned weights_llrs are tuned to the training distribution's per-qubit rates; fresh CER priors track the drift. Uniform-prior models should suffer the most because they have no per-qubit lever to adjust — only the trained, now-mismatched, weights. The per-qubit-prior config keeps a knob the operator can turn without retraining.

## 13. Where to go next

What we *didn't* show but the package supports out of the box:

- **Quantum / CSS codes**: pass a separate `parity_check_matrix_dual` containing the logical operators so `check_bp_solutions` validates *logical equivalence*, not exact recovery. The pipeline is otherwise identical.
- **File-backed CER data**: `load_base_bp_model(parity_check_file, logicals_file, n_layers, correlation_strengths_file=...)` is the production path used for the BB-code experiments. It reads a CER text file with `qubit : rate` and `(i,j) : strength` lines and wires the connectivity + correlation strengths into the base.
- **Bigger codes**: the same `train_nachmani_neuralbp` works on the 72-qubit and 90-qubit Bivariate-Bicycle (BB) codes used in the head-to-head Julia comparison. The `scripts/head_to_head_explicit_compare.py` runner is the production entry point.

Things worth tweaking when you train on your own code:

- **`n_layers`** controls expressive depth and runtime; 5–10 is typical for short codes.
- **`initial_conditions_scale`** in the weight init: small perturbations around 1 (≤ 0.3) train more stably than fully random weights because they start near the BP fixed point.
- **Annealing schedules** for `llr_certainty_importance` and `sparsity_importance` matter more than you would expect — start them small and tune downward as the base loss takes over.
- **Loss aggregation**: the `linear_ramp` mode (used here) is the Julia default and tends to be the most stable starting point; `softmin` is more aggressive about chasing the best individual layer.

All of these levers are exposed on `TrainingConfig` and `NachmaniNeuralBP`.

## 14. Handoff to NN experts: where to push the architecture

The current `NachmaniNeuralBP` is already a `torch.nn.Module`, but it is a deliberately hand-coded unrolled-BP network: per-edge **scalar** weights, the *exact* BP message non-linearity (`log tanh` / `atanh exp`), and the Tanner-graph topology baked in via `index_add_` kernels. That conservative design is intentional — it gives an apples-to-apples comparison with classical BP, with the all-ones-weights equivalence we verified in §8 — but it leaves most of the standard NN toolbox on the table. A few natural moves for an NN-expert collaborator:

### Reframing in higher-level PyTorch idiom

The decoder is naturally a **graph neural network** with two node types (check / variable) and edges given by the parity-check matrix. `torch_geometric.nn.MessagePassing` gives you the exact propagate / message / aggregate split that BP implements: `c2v_to_v2c` and `v2c_to_c2v` become two `MessagePassing` layers with custom `message()` functions, and the existing `index_add_`-based sparse multiply is what `torch_geometric` expresses as `scatter_add` along an edge index. A faithful rewrite would (a) make the architecture readable to someone who hasn't been steeped in BP, (b) plug the model into the broader GNN ecosystem (visualizations, profilers, distributed training), and (c) be structurally identical to the current implementation — so any subsequent change in performance comes from architecture rather than from substituting kernels.

### Research directions, in roughly increasing order of departure from BP

1. **Per-edge MLPs instead of scalar weights.** Replace each scalar message weight with a small MLP taking the message magnitude (and optionally local neighborhood context) as input. One step beyond the Nachmani scalar parameterization; modest parameter increase, often the single largest accuracy gain reported in the neural-decoder literature.
2. **Learned non-linearity.** The fixed BP non-linearity was chosen for analytic exactness on trees, not for finite Tanner graphs with cycles. A small per-layer learned activation (or a `tanh`-modulated GRU-style update) gives the network freedom to soften or sharpen messages where short cycles bite.
3. **Recurrent (shared-weight) layers.** Today every layer has its own weights. Sharing across layers turns the architecture into a recurrent decoder — far fewer parameters, often better generalization, and variable-depth inference for free (run for as many iterations as the latency budget allows).
4. **Attention over edges.** A check node currently weights all incoming V2C messages with an equal-weighted product. A graph-attention layer could let the decoder up-weight the most informative messages per check — a natural fit for short-cycle codes where some neighborhoods are more trustworthy than others.
5. **Conditional / hyper-decoders.** Feed CER metadata (per-qubit rates, pair strengths) as *inputs* to the network rather than baking them into priors and loss. A single trained decoder could then handle many noise profiles without retraining — directly addresses the device-drift scenario from §13.
6. **Pre-training across codes.** Pre-train on a curriculum of randomly generated parity-check matrices, then fine-tune for the specific code of interest. This is the natural path toward a "foundation-model" decoder that transfers across code families.

The cleanest entry point is probably **(1) + (3)**: port the decoder to `torch_geometric.nn.MessagePassing` with per-edge MLPs and layer-shared weights. That keeps the comparison head-to-head with the Nachmani baseline trained in §10 — same data, same evaluation — while opening the door to everything else above.